# Training a model for the CM4AI DREAM Challenge

This tutorial walks through training a model on challenge data, checking how well it does, and turning it into a submission. The model under test is **BLAST-ProtiCelli**, the challenge's example model; each step shows where your own model would take its place.

BLAST-ProtiCelli takes a protein's amino acid sequence and three reference images of a cell, and predicts an image of where that protein sits in the cell. How it does that is not the subject here; the repository's README has the details.

![Tutorial flow: steps 1 to 5 train a model in this notebook and repeat until it is ready; step 6 submits it](../docs/tutorial_flow.svg)

**Phase A, in this notebook: train a model**

1. **Set up:** settings (1a), install the model (1b), check the GPU (1c), choose the data (1d), and get the starting weights (1e).
2. **Format the data:** download training cells and convert them to the form the model reads.
3. **Train:** train the model further on the new cells, then package it as a weights file.
4. **Predict** held-out cells with the starting and the trained weights.
5. **Evaluate** both with `cm4ai-evaluate`, then decide: add data, change the model, or submit.

**Phase B, on the command line: submit.** The weights file from step 3 goes into the model's workflow, `blast_proticelli.wdl`, unchanged; a separate tutorial covers a test run on one or two cells.

The training here is deliberately small. The goal is to show where your model fits, not to improve the example model; see the notes in step 3 before reading anything into the results.

**Runs on** an AnVIL (Terra) Jupyter environment with a GPU, and also on a local machine or cluster.


## 1. Set up

### 1a. Settings

The notebook's control panel. Start with `RUN_MODE = "quick"`, which runs every step on a small sample to check that everything works; switch to `"full"` when you are ready to train for real. The training and prediction values belong to the example model: with your own model, these are the values you would change.

Everything the notebook creates goes under `WORK_DIR`, by default the repository's `training/work` folder, so deleting that folder resets the notebook.


In [ ]:
import os
from pathlib import Path

RUN_MODE = "quick"          # "quick": a small run that checks every step; "full": a real training run
SEED = 1                    # same seed, same antibodies, split and predictions
WORK_DIR = None             # everything this notebook writes; None: the repository's training/work folder

# This repository's code. Used only when no copy is found next to the notebook or in the home folder
# (for example, the notebook uploaded on its own to a Terra workspace): the repository is then cloned
# into the home folder.
REPO_URL = "https://github.com/cm4ai-anvil-dream-challenge/blast-proticelli.git"
REPO_BRANCH = "main"

# Optional: an existing BLAST-ProtiCelli weights zip (for example blast_proticelli_weights_v2.zip).
# Its model folders are reused instead of downloading them again (about 6.5 GB).
# Read from the environment variable BLAST_PROTICELLI_WEIGHTS_ZIP if set, or type a path here.
# A gs:// path (for example a zip in an AnVIL workspace bucket) is copied once into WORK_DIR/downloads.
# The UniProt reference data is always rebuilt, so the new weights file gets the current gene map.
EXISTING_WEIGHTS_ZIP = os.environ.get("BLAST_PROTICELLI_WEIGHTS_ZIP")

# Training (step 3), values for the example model. Training all of it (about 460M parameters) needs
# roughly 8 GB of GPU memory before images; lower BATCH_SIZE if training runs out of memory.
EPOCHS = {"quick": 1, "full": 20}[RUN_MODE]
BATCH_SIZE = 4
LEARNING_RATE = 1e-5        # peak; the model's own default is 1e-4, gentler here for a small dataset
RETRAIN = False             # False: reuse fine-tuned weights from an earlier run in WORK_DIR/finetune, if present

# Prediction (step 4): diffusion steps per prediction, and how many held-out cells to compare (None: all)
PREDICTION_STEPS = {"quick": 20, "full": 50}[RUN_MODE]
COMPARE_CELLS = {"quick": 12, "full": None}[RUN_MODE]

print(f"Settings: RUN_MODE={RUN_MODE}, epochs={EPOCHS}, batch size={BATCH_SIZE}, seed={SEED}")


### 1b. Install the model

Installs the example model's code and everything it needs, once per environment. In a submission, the model's Docker image does this; here, without Docker, the notebook does it.

The code is this repository, BLAST-ProtiCelli. `pip install` also brings in its Python dependencies, including the ProtiCelli package it builds on (pinned in `pyproject.toml`). On AnVIL this takes several minutes, mostly for PyTorch; afterwards, restart the kernel and run from the top.

The notebook normally sits in the `training/` folder of a clone of the repository:

```bash
git clone https://github.com/cm4ai-anvil-dream-challenge/blast-proticelli.git
```

Opened anywhere else (for example uploaded to a Terra workspace's Analyses tab), it uses a `blast-proticelli` folder in the home directory, cloning `REPO_BRANCH` of `REPO_URL` there if it is missing.


In [ ]:
import subprocess, sys

def is_repo(d):
    return (d / "pyproject.toml").is_file() and (d / "training" / "build_weights_zip.sh").is_file()

REPO_DIR = next((d for d in [Path.cwd().parent, Path.cwd() / "blast-proticelli", Path.home() / "blast-proticelli"]
                 if is_repo(d)), None)
if REPO_DIR is None:
    REPO_DIR = Path.home() / "blast-proticelli"
    print(f"No clone of the repository found; cloning {REPO_URL} ({REPO_BRANCH}) into {REPO_DIR}")
    proc = subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
                          capture_output=True, text=True, env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})
    if proc.returncode != 0:
        raise RuntimeError(f"Could not clone {REPO_URL} ({REPO_BRANCH}). A private repository needs credentials; "
                           f"check REPO_URL and REPO_BRANCH in section 1.\n{proc.stderr.strip()}")
    assert is_repo(REPO_DIR), f"{REPO_DIR} does not look like the blast-proticelli repository"
print("Repository:", REPO_DIR)

WORK_DIR = Path(WORK_DIR).expanduser().resolve() if WORK_DIR else REPO_DIR / "training" / "work"
DOWNLOAD_DIR = WORK_DIR / "downloads"    # starting weights, reference data
print("Work folder:", WORK_DIR)

from importlib import metadata

try:
    print("Model already installed: blast-proticelli", metadata.version("blast-proticelli"))
except metadata.PackageNotFoundError:
    # pip's output is printed here as it arrives (several minutes, mostly PyTorch); lines about packages
    # that are already installed are left out
    proc = subprocess.Popen([sys.executable, "-m", "pip", "install", "--progress-bar", "off", str(REPO_DIR)],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        if not line.startswith("Requirement already satisfied"):
            print(line, end="", flush=True)
    if proc.wait() != 0:
        raise RuntimeError("pip install failed; see the output above")
    print("Installed. Restart the kernel and run from the top.")


The example model also uses BLAST+, NCBI's sequence search program, to identify proteins from their sequence. BLAST+ is a compiled program rather than a Python package, so pip cannot install it; `training/install_blast.sh` does, on Linux (including AnVIL), pinned to the same version as the Docker image. On a Mac, install it first with `conda install -c bioconda blast`.


In [ ]:
import shutil

if not (shutil.which("blastp") and shutil.which("makeblastdb")):
    proc = subprocess.run(["bash", str(REPO_DIR / "training" / "install_blast.sh"), str(WORK_DIR / "tools")],
                          capture_output=True, text=True)
    if proc.returncode != 0:
        raise RuntimeError(proc.stderr.strip())
    os.environ["PATH"] = proc.stdout.strip().splitlines()[-1] + os.pathsep + os.environ["PATH"]
print(subprocess.run(["blastp", "-version"], capture_output=True, text=True).stdout.splitlines()[0])


### 1c. Check the GPU

Training needs a GPU to finish in reasonable time. On AnVIL, the GPU is chosen when the cloud environment is created; if this cell reports CPU, recreate the environment with a GPU attached.


In [ ]:
import torch

if torch.cuda.is_available():
    DEVICE = "cuda"
    print("CUDA GPU:", torch.cuda.get_device_name(0), f"({torch.cuda.get_device_properties(0).total_memory / 2**30:.0f} GiB)")
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    DEVICE = "mps"
    print("Apple GPU (MPS)")
else:
    DEVICE = "cpu"
    print("No GPU found: training will be very slow. Fine for checking the steps run, not for real training.")
print("PyTorch", torch.__version__)

### 1d. Choose the data

*Being rewritten next: this step will choose the data source (the AnVIL bucket, Hugging Face, the CFDE Cloud Workbench, or a folder of your own). Until then it holds the previous HPA-train settings, so the rest of the notebook still runs.*


In [ ]:
# Training data: HPA-train_v1.0 (step 2)
HPA_TRAIN_URI = "gs://fc-7533ae9b-e06e-4297-9bd5-a4dbc0895420/HPA-train_v1.0"
# The bucket is requester pays: downloads are billed to this Google Cloud project.
# Read from the environment variable GOOGLE_PROJECT, which AnVIL notebook sessions set to the workspace's
# project. Elsewhere, set GOOGLE_PROJECT yourself before starting Jupyter (keeps project IDs out of the notebook).
BILLING_PROJECT = os.environ.get("GOOGLE_PROJECT")
# Optional: a folder already holding the manifest and/or extracted antibody folders, used before downloading
LOCAL_HPA_DIR = None
HPA_DIR = WORK_DIR / "hpa"  # manifest and extracted antibody folders

N_TRAIN_ANTIBODIES = {"quick": 10, "full": 500}[RUN_MODE]
MAX_TRAIN_CELLS = {"quick": 200, "full": None}[RUN_MODE]     # None: every cell of the training antibodies
N_TEST_ANTIBODIES = {"quick": 3, "full": 50}[RUN_MODE]
SPLIT_BY = "antibody"       # "antibody", "fov" or "cell": which cells may not be split between training and held-out sets

print(f"Data: {HPA_TRAIN_URI}, {N_TRAIN_ANTIBODIES} training and {N_TEST_ANTIBODIES} held-out antibodies")


### 1e. Get the starting weights

Training starts from the example model's published weights, and the model's weights file also carries the reference data it uses to identify proteins. Two downloads, both kept under `WORK_DIR/downloads`:

* **Starting weights:** ProtiCelli's published weights from the Lundberg lab (`checkpoint/` and `vae/`, about 6.5 GB), unless `EXISTING_WEIGHTS_ZIP` is set or a previous run already fetched them.
* **Reference data:** the reviewed human proteins from UniProt as a BLAST database, and a gene map from UniProt accessions and Ensembl protein IDs to HGNC symbols. These are built by `build_human_blast_db.sh` and `build_human_gene_map.sh` in this folder, which can also be run on their own. They always rebuild from the current UniProt release, which is recorded in each zip's `PROVENANCE.txt`.

A download that fails stops the cell with the reason; rerunning the cell picks up where it left off.


In [ ]:
import shutil
import zipfile

def local_weights_zip(src):
    # A local path is used as is; a gs:// path is copied once into DOWNLOAD_DIR.
    if not str(src).startswith("gs://"):
        return Path(src).expanduser()
    local = DOWNLOAD_DIR / str(src).rstrip("/").split("/")[-1]
    if local.is_file():
        print("Reusing", local)
        return local
    DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
    partial = local.with_name(local.name + ".part")
    billing = [f"--billing-project={BILLING_PROJECT}"] if BILLING_PROJECT else []
    print(f"Copying {src} to {local} ...", flush=True)
    proc = subprocess.run(["gcloud", "storage", "cp", *billing, str(src), str(partial)], capture_output=True, text=True)
    if proc.returncode != 0:
        raise RuntimeError(f"Could not copy {src}\n{proc.stderr.strip()}")
    partial.rename(local)
    return local

if EXISTING_WEIGHTS_ZIP:
    EXISTING_WEIGHTS_ZIP = str(local_weights_zip(EXISTING_WEIGHTS_ZIP))   # section 6 also reads it, as the baseline

sys.path.insert(0, str(REPO_DIR))      # use this repository's code, not an older installed copy
from proticelli.utils.download import checkpoint_ready, vae_ready, download_checkpoints

PROTICELLI_DIR = DOWNLOAD_DIR / "proticelli"     # will hold checkpoint/ and vae/
TRAINING_SCRIPTS = REPO_DIR / "training"

def proticelli_weights_ready(d):
    # fit() starts from checkpoint/unet_ema; inference loads checkpoint/unet. Require both, plus the VAE.
    ema = d / "checkpoint" / "unet_ema"
    return checkpoint_ready(d) and vae_ready(d) and (ema / "config.json").is_file()

if proticelli_weights_ready(PROTICELLI_DIR):
    print("Reusing ProtiCelli weights in", PROTICELLI_DIR)
elif EXISTING_WEIGHTS_ZIP:
    print("Extracting ProtiCelli weights from", EXISTING_WEIGHTS_ZIP)
    with zipfile.ZipFile(Path(EXISTING_WEIGHTS_ZIP).expanduser()) as zf:
        members = [n for n in zf.namelist() if n.startswith(("checkpoint/unet/", "checkpoint/unet_ema/", "vae/"))]
        zf.extractall(PROTICELLI_DIR, members=members)
else:
    download_checkpoints(PROTICELLI_DIR)

assert proticelli_weights_ready(PROTICELLI_DIR), f"ProtiCelli weights in {PROTICELLI_DIR} are incomplete"
print("ProtiCelli weights ready")

In [ ]:
REF_DIR = DOWNLOAD_DIR / "reference"
BLAST_DB_DIR = REF_DIR / "human_blast_db"
GENE_MAP_DIR = REF_DIR / "human_gene_map"

def build_reference(script, out_dir, product):
    if (out_dir / product).is_file():
        print(f"Reusing {out_dir / product}")
        return
    print(f"Running {script} (downloads from UniProt; a few minutes)")
    proc = subprocess.run(["bash", str(TRAINING_SCRIPTS / script), str(out_dir)], capture_output=True, text=True)
    print(proc.stdout[-2000:])
    if proc.returncode != 0:
        raise RuntimeError(f"{script} failed:\n{proc.stderr[-2000:]}")

build_reference("build_human_blast_db.sh", BLAST_DB_DIR, "human_blast_db.zip")
build_reference("build_human_gene_map.sh", GENE_MAP_DIR, "human_gene_map.zip")
for d in (BLAST_DB_DIR, GENE_MAP_DIR):
    print((d / "PROVENANCE.txt").read_text())

## 4. The training data: HPA-train

**HPA-train_v1.0** holds 1.25 million single-cell images from the [Human Protein Atlas](https://www.proteinatlas.org/) subcellular section: 15,674 antibodies against 12,698 proteins, imaged in 39 cell lines. Every cell has the same five 512 x 512 PNGs the challenge uses (protein, nucleus, microtubules, ER, and a cell mask), and every antibody has a FASTA file with its target's amino acid sequence.

How it is stored:

* `hpa_train_manifest.csv.gz`: one row per cell, with its antibody, gene, Ensembl IDs, cell line and the path of each file. **Take identifiers and paths from these columns, never from file names** (cell line names contain hyphens).
* `tars/<first 5 characters of the antibody ID>/<antibody ID>.tar`: one archive per antibody (median 34 MB; 609 GB in all), holding a folder of that antibody's cells.
* `LICENSE.txt`: terms of use for the images.

The bucket is *requester pays*: whoever downloads pays the transfer cost, billed to `BILLING_PROJECT`. On AnVIL this is the workspace's own project, found automatically. Elsewhere, set the environment variable `GOOGLE_PROJECT` to a Google Cloud project you can bill to, and sign in with `gcloud auth login`.

**An important caveat.** ProtiCelli was trained on Human Protein Atlas images, and its developers confirm these overlap with HPA-train. Fine-tuning on it here shows *how* a team trains, not a way to improve ProtiCelli. The public CM4AI images of the challenge's own cell lines (MDA-MB-468 and KOLF2.1J), released separately, are the data for genuine fine-tuning.

In [ ]:
import json
import tarfile
import numpy as np
import pandas as pd

HPA_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_HPA = Path(LOCAL_HPA_DIR).expanduser() if LOCAL_HPA_DIR else None

def gs_copy(uri, dest):
    # Download one object from the requester pays bucket, billed to BILLING_PROJECT.
    if not BILLING_PROJECT:
        raise RuntimeError(
            "The HPA-train bucket is requester pays, so downloads need a Google Cloud project to bill: "
            "set the environment variable GOOGLE_PROJECT before starting Jupyter, or BILLING_PROJECT in section 1 "
            "(on AnVIL it is set automatically)."
        )
    if shutil.which("gcloud"):
        cmd = ["gcloud", "storage", "cp", f"--billing-project={BILLING_PROJECT}", uri, str(dest)]
    else:
        cmd = ["gsutil", "-q", "-u", BILLING_PROJECT, "cp", uri, str(dest)]
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.returncode != 0:
        raise RuntimeError(f"Download failed: {uri}\n{proc.stderr.strip()}")

def hpa_file(name):
    # A top-level file of HPA-train: from LOCAL_HPA_DIR if it is there, otherwise downloaded once into HPA_DIR.
    for root in [LOCAL_HPA, HPA_DIR]:
        if root and (root / name).is_file():
            return root / name
    gs_copy(f"{HPA_TRAIN_URI}/{name}", HPA_DIR / name)
    return HPA_DIR / name

# Terms of use for the images
try:
    print(hpa_file("LICENSE.txt").read_text())
except RuntimeError as err:
    print(f"LICENSE.txt not shown: {err}")

In [ ]:
# The manifest: 1.25 million rows, about 36 MB compressed. Also accepts an uncompressed copy in LOCAL_HPA_DIR.
local_plain = LOCAL_HPA / "hpa_train_manifest.csv" if LOCAL_HPA else None
manifest_path = local_plain if local_plain and local_plain.is_file() else hpa_file("hpa_train_manifest.csv.gz")
manifest = pd.read_csv(manifest_path, dtype=str, keep_default_na=False)
print(f"{manifest_path}: {len(manifest):,} cells, {manifest['antibody_id'].nunique():,} antibodies, "
      f"{manifest['gene_name'].nunique():,} genes, {manifest['cell_line'].nunique()} cell lines\n")

per_antibody = manifest.groupby("antibody_id").size()
print(f"Cells per antibody: median {per_antibody.median():.0f}, range {per_antibody.min()} to {per_antibody.max()}")
print(f"Cells per field of view: median {manifest.groupby('fov_key').size().median():.0f}\n")
print("Cells per cell line (top 10):")
print(manifest["cell_line"].value_counts().head(10).to_string())
manifest.head(3)

### 4a. Downloading antibodies

`fetch_antibodies()` downloads and extracts the tars for a list of antibodies into `WORK_DIR/hpa`, skipping any already there (or in `LOCAL_HPA_DIR`), and checks that every file the manifest lists for them arrived. Section 4b uses it for the training and held-out antibodies; here it fetches one small antibody to look at.

In [ ]:
def antibody_root(antibody_id):
    # Folder holding the extracted <antibody_id>/ folder, or None if it is not present yet.
    for root in [LOCAL_HPA, HPA_DIR]:
        if root and (root / antibody_id).is_dir():
            return root
    return None

def fetch_antibodies(antibody_ids):
    rows = manifest[manifest["antibody_id"].isin(antibody_ids)]
    tars = rows.drop_duplicates("antibody_id").set_index("antibody_id")["tar_file"]
    missing = sorted(set(antibody_ids) - set(tars.index))
    if missing:
        raise KeyError(f"Not in the manifest: {missing}")
    staging = HPA_DIR / "_tars"
    staging.mkdir(parents=True, exist_ok=True)
    fetched = 0
    for i, (antibody_id, tar_name) in enumerate(tars.items(), 1):
        if antibody_root(antibody_id):
            continue
        tar_path = staging / tar_name
        gs_copy(f"{HPA_TRAIN_URI}/tars/{antibody_id[:5]}/{tar_name}", tar_path)
        with tarfile.open(tar_path) as t:
            try:
                t.extractall(HPA_DIR, filter="data")   # refuses unsafe paths (Python 3.12, and 3.11.4 or later)
            except TypeError:
                t.extractall(HPA_DIR)
        tar_path.unlink()
        fetched += 1
        print(f"  [{i}/{len(tars)}] {antibody_id}")
    # Every file the manifest lists for these antibodies must now exist
    path_columns = ["protein_path", "nucleus_path", "microtubules_path", "er_path", "mask_path", "sequence_path"]
    absent = [p for _, r in rows.iterrows() for p in (antibody_root(r["antibody_id"]) / r[c] for c in path_columns) if not p.is_file()]
    if absent:
        raise FileNotFoundError(f"{len(absent)} files listed in the manifest are missing, for example {absent[0]}")
    print(f"{len(tars)} antibodies ready ({fetched} downloaded), {len(rows)} cells")
    return rows

def cell_files(row):
    # Absolute paths of one cell's files, from its manifest row.
    root = antibody_root(row["antibody_id"])
    return {c[: -len("_path")]: root / row[c] for c in ["protein_path", "nucleus_path", "microtubules_path", "er_path", "mask_path", "sequence_path"]}

In [ ]:
import matplotlib.pyplot as plt
from skimage.io import imread

def scaled(img):
    # 0 to 1, stretched to the image's 99.9th percentile so dim stains are visible
    img = img.astype(np.float32)
    top = np.percentile(img, 99.9)
    return np.clip(img / top, 0, 1) if top > 0 else img

def reference_composite(microtubules, nucleus, er):
    # The usual immunofluorescence colors: nucleus blue, microtubules red, ER yellow
    mt, nu, e = scaled(microtubules), scaled(nucleus), scaled(er)
    return np.clip(np.stack([mt + e, e, nu], axis=-1), 0, 1)

def protein_green(protein, top=None):
    # Protein in green on black; pass top to put several images on one scale
    p = protein.astype(np.float32) / (top or max(np.percentile(protein, 99.9), 1))
    return np.stack([np.zeros_like(p), np.clip(p, 0, 1), np.zeros_like(p)], axis=-1)

EXAMPLE_ANTIBODY = "HPA073913"    # a small one: 9 cells, about 1 MB
example = fetch_antibodies([EXAMPLE_ANTIBODY]).iloc[0]
files = cell_files(example)

img = {ch: imread(files[ch]) for ch in ["microtubules", "nucleus", "er", "protein", "mask"]}
fig, axes = plt.subplots(1, 3, figsize=(12, 4.2))
axes[0].imshow(reference_composite(img["microtubules"], img["nucleus"], img["er"]))
axes[0].set_title("reference: nucleus (blue), microtubules (red), ER (yellow)", fontsize=9)
axes[1].imshow(protein_green(img["protein"]))
axes[1].set_title(f"protein: {example['gene_name']}", fontsize=9)
axes[2].imshow(protein_green(img["protein"]))
axes[2].contour(img["mask"] > 0, levels=[0.5], colors="white", linewidths=0.8)
axes[2].set_title("protein, with the cell mask outlined", fontsize=9)
for ax in axes:
    ax.axis("off")
fig.suptitle(f"{example['image_key']} ({example['cell_line']}): the reference channels go in, the protein image is what a model predicts")
plt.show()
print(files["sequence"].read_text()[:300])

### 4b. Matching proteins and cell lines to ProtiCelli

ProtiCelli learns one embedding per protein and per cell line, so `fit()` needs every training cell's protein and cell line in ProtiCelli's own vocabularies (`antibody_map.pkl` and `cell_line_map.pkl` in the installed package). HPA-train names them almost the same way, with two differences handled here:

* **Antibodies against several related proteins.** HPA lists their genes as `TPM1;TPM3`; ProtiCelli has one entry for the group, `TPM1,TPM3`. They are matched as sets of names.
* **Cell line punctuation.** HPA writes `HUVEC/TERT2` where ProtiCelli writes `HUVEC TERT2`. This matters: `fit()` treats a cell line it does not recognize as unknown, without an error.

Antibodies whose protein is not in the vocabulary cannot be used for training as is. Adding them means enlarging the model's protein embedding table and training the new rows: one of the places a team could take this further.

In [ ]:
import pickle
from importlib import resources
import protein_predictor as pp

proticelli_data = resources.files("proticelli") / "data"
with open(proticelli_data / "antibody_map.pkl", "rb") as f:
    protein_keys = list(pickle.load(f))
with open(proticelli_data / "cell_line_map.pkl", "rb") as f:
    proticelli_cell_lines = set(pickle.load(f))
vocabulary = pp.Vocabulary(protein_keys)
groups = {frozenset(t.strip() for t in k.split(",")): k for k in protein_keys}

def protein_key(gene_name):
    # ProtiCelli's vocabulary entry for an HPA gene name, or None.
    if ";" in gene_name:
        return groups.get(frozenset(g.strip() for g in gene_name.split(";")))
    return vocabulary.resolve(gene_name)

def cell_line_key(name):
    # ProtiCelli's name for an HPA cell line, or None.
    for candidate in (name, name.replace("/", " ")):
        if candidate in proticelli_cell_lines:
            return candidate
    return None

antibodies = manifest.drop_duplicates("antibody_id")[["antibody_id", "gene_name"]].copy()
antibodies["protein_key"] = antibodies["gene_name"].map(protein_key)
cell_line_names = {c: cell_line_key(c) for c in manifest["cell_line"].unique()}

usable = antibodies["protein_key"].notna()
print(f"Proteins: {usable.sum():,} of {len(antibodies):,} antibodies match ProtiCelli's vocabulary "
      f"({(antibodies.loc[usable, 'gene_name'] != antibodies.loc[usable, 'protein_key']).sum()} of them as a group of genes)")
print(f"Cell lines: {sum(v is not None for v in cell_line_names.values())} of {len(cell_line_names)} match; "
      f"renamed: {({k: v for k, v in cell_line_names.items() if v and v != k})}")
unmatched_lines = sorted(k for k, v in cell_line_names.items() if v is None)
if unmatched_lines:
    print("Cell lines ProtiCelli does not know (trained as unknown):", unmatched_lines)

not_in_vocabulary = antibodies[~usable]
not_in_vocabulary.to_csv(WORK_DIR / "antibodies_not_in_vocabulary.csv", index=False)
print(f"\n{len(not_in_vocabulary)} antibodies cannot be used (listed in {WORK_DIR / 'antibodies_not_in_vocabulary.csv'}), for example:")
not_in_vocabulary[["antibody_id", "gene_name"]].head(8)

### 4c. Choosing antibodies and splitting

`N_TRAIN_ANTIBODIES + N_TEST_ANTIBODIES` antibodies are drawn at random (seeded by `SEED`) from those ProtiCelli knows. Their cells are then split into a training set and a held-out set, which section 6 uses to compare the original and fine-tuned weights.

**How to split** (`SPLIT_BY`) is a real choice. Cells of one antibody, and above all cells from one field of view, look much alike: same staining, same imaging session, often neighboring cells. If near copies end up on both sides, the held-out score partly measures memory. So:

* `"antibody"` (default): every cell of an antibody is on the same side, so held-out proteins were never seen in training. This matches the challenge, which asks for proteins in new cells.
* `"fov"`: fields of view are kept together, but an antibody can be on both sides. (In HPA-train a field of view never holds two antibodies.)
* `"cell"`: cells are split individually. Fastest to learn from, easiest to fool yourself with.

In [ ]:
rng = np.random.default_rng(SEED)
pool = antibodies.loc[usable, "antibody_id"].to_numpy()
chosen = rng.choice(pool, size=N_TRAIN_ANTIBODIES + N_TEST_ANTIBODIES, replace=False)

cells = manifest[manifest["antibody_id"].isin(chosen)].copy()
cells["protein_key"] = cells["antibody_id"].map(antibodies.set_index("antibody_id")["protein_key"])
cells["proticelli_cell_line"] = cells["cell_line"].map(cell_line_names)

if SPLIT_BY == "antibody":
    held_out_groups = set(chosen[:N_TEST_ANTIBODIES])
    group_column = "antibody_id"
elif SPLIT_BY in ("fov", "cell"):
    group_column = "fov_key" if SPLIT_BY == "fov" else "image_key"
    group_ids = cells[group_column].unique()
    rng.shuffle(group_ids)
    n_held_out = round(len(group_ids) * N_TEST_ANTIBODIES / (N_TRAIN_ANTIBODIES + N_TEST_ANTIBODIES))
    held_out_groups = set(group_ids[:n_held_out])
else:
    raise ValueError(f'SPLIT_BY must be "antibody", "fov" or "cell", not {SPLIT_BY!r}')
cells["split"] = np.where(cells[group_column].isin(held_out_groups), "test", "train")

print(f"{len(chosen)} antibodies, {len(cells):,} cells, split by {SPLIT_BY}:")
print(cells.groupby("split").agg(cells=("image_key", "size"), antibodies=("antibody_id", "nunique"),
                                 cell_lines=("cell_line", "nunique")).to_string(), "\n")

fetch_antibodies(list(chosen))
cells.to_csv(WORK_DIR / "training_cells.csv", index=False)
print(f"Written to {WORK_DIR / 'training_cells.csv'}")
cells.drop_duplicates("antibody_id")[["antibody_id", "gene_name", "protein_key", "split"]].sort_values("split")

### 4d. Building the training images

`fit()` reads each training cell as one 4-channel TIFF. Each is assembled and normalized with ProtiCelli's own `ChannelAssembler` and `ImageNormalizer`, the same steps the baseline applies to its inputs at prediction time, so training and prediction see identically scaled images (values from -1 to 1).

* **Channel order** is ProtiCelli's: microtubules, protein, nucleus, ER. (The `fit()` docstring names the channels differently; its code uses positions 0, 2 and 3 as the reference channels and 1 as the protein to learn, which matches this order.)
* **Compression.** The TIFFs are stored with lossless zlib compression, about 0.2 MB per cell rather than 4 MB, so `"full"` runs fit on an ordinary disk.
* **Held-out cells are not converted.** Section 6 runs them through the normal prediction pipeline from the original PNGs.

In [ ]:
import tifffile
from image_scaling import read_png       # checks the challenge format: single-channel 8-bit PNG, 512 x 512
from proticelli.data import ChannelAssembler, ImageNormalizer

TRAIN_DIR = WORK_DIR / "train_tiff"
TRAIN_DIR.mkdir(parents=True, exist_ok=True)
assembler, normalizer = ChannelAssembler(has_protein=True), ImageNormalizer(bit_depth=8)
CHANNELS = ("microtubules", "nucleus", "er", "protein")

train_cells = cells[cells["split"] == "train"]
if MAX_TRAIN_CELLS and len(train_cells) > MAX_TRAIN_CELLS:
    train_cells = train_cells.sample(MAX_TRAIN_CELLS, random_state=SEED).sort_index()
    print(f"Using {MAX_TRAIN_CELLS} of the training antibodies' cells (MAX_TRAIN_CELLS)")
built = 0
for i, (_, row) in enumerate(train_cells.iterrows(), 1):
    out = TRAIN_DIR / f"{row['image_key']}.tiff"
    if out.exists():
        continue
    files = cell_files(row)
    raw = assembler.transform({ch: read_png(files[ch]) for ch in CHANNELS})
    stack = normalizer.transform(raw).astype(np.float32)
    # fit() rescales any image whose maximum is above 1 as if it were raw counts; normalized images never are
    assert stack.shape == (512, 512, 4) and stack.min() >= -1 and stack.max() <= 1, row["image_key"]
    tifffile.imwrite(out, stack, compression="zlib")
    built += 1
    if i % 200 == 0:
        print(f"  {i} of {len(train_cells)}")

TRAIN_FILES = [f"{k}.tiff" for k in train_cells["image_key"]]
TRAIN_PROTEINS = train_cells["protein_key"].tolist()
TRAIN_CELL_LINES = train_cells["proticelli_cell_line"].tolist()
size_mb = sum((TRAIN_DIR / f).stat().st_size for f in TRAIN_FILES) / 1e6
print(f"{len(TRAIN_FILES)} training images in {TRAIN_DIR} ({built} built now, {size_mb:.0f} MB), "
      f"{train_cells['protein_key'].nunique()} proteins, {train_cells['cell_line'].nunique()} cell lines")

## 5. Fine-tune

`Model.fit()` trains ProtiCelli on the training images, starting from the published weights. Every step, a batch of real protein images is corrupted with noise of a random strength, and the model learns to recover them given the three reference channels, the protein's name and the cell line. The plot below updates as training runs: each point is one step's loss, and the line is a running average.

**This is where a team's own model goes.** Everything before this cell prepares data in the challenge's format; everything after it packages and evaluates whatever weights this cell produces.

**Reading the plot.** The per-step loss jumps around because each step draws a different noise strength; the running average is the thing to watch. A falling average means the model is fitting the training images, not that its predictions for new images got better; section 6 compares the original and fine-tuned weights on held-out images for that. In `"quick"` mode, expect a short, noisy curve.

**Weights.** ProtiCelli ships two copies of its weights: `checkpoint/unet`, which prediction loads, and `checkpoint/unet_ema`, a smoothed running average from its original training. `fit()` starts from `unet_ema` and, as configured here (EMA averaging off), saves the fine-tuned weights to a folder named `unet_ema`. Section 6 packages them as `checkpoint/unet` so prediction uses them. Whether to keep an EMA copy during fine-tuning is a choice teams can revisit.

**What to expect.** These HPA images overlap ProtiCelli's own training data (section 4), so the model has likely seen these cells or their neighbors before; expect little change, and read the held-out comparison in section 6 with that in mind. Each cell's cell line is passed to `fit()`, so the model's cell line embeddings are trained too.

**How long.** Run time is steps x seconds per step, and steps = epochs x training cells / `BATCH_SIZE`. Measured so far: about 17 seconds per step on an Apple M1-class laptop GPU at batch size 4, so `"quick"` mode (200 cells, 50 steps) takes about 15 minutes. Datacenter GPUs are much faster; the cell prints its own timing when it finishes.

**Memory.** Fine-tuning every weight of ProtiCelli needs roughly 8 GB of GPU memory. On a laptop, that comes out of the same memory as everything else: close large applications first, and if Activity Monitor (Memory tab) shows swap growing, training slows down many times over. A smaller `BATCH_SIZE` helps a little; the weights and optimizer state are most of the memory.

**Reusing a run.** With `RETRAIN = False`, an earlier run's fine-tuned weights in `WORK_DIR/finetune` are reused: this cell then skips training and redraws that run's loss curve, so the notebook can be rerun from the top (after a kernel restart, say) without training again.

In [ ]:
import datetime
import math
from IPython.display import display
from proticelli import Model
import proticelli._training as proticelli_training

import time

FINETUNE_DIR = WORK_DIR / "finetune"
RUN_RECORD = FINETUNE_DIR / "training_run.json"
LOSS_FILE = FINETUNE_DIR / "training_loss.csv"
steps_per_epoch = math.ceil(len(TRAIN_FILES) / BATCH_SIZE)
total_steps = EPOCHS * steps_per_epoch
warmup_steps = max(1, total_steps // 10)   # fit()'s default of 500 would outlast a short run

fig, ax = plt.subplots(figsize=(9, 3.5))
plot_handle = display(fig, display_id=True)
plt.close(fig)
step_losses = []

def redraw(title=None):
    ax.clear()
    ax.plot(range(1, len(step_losses) + 1), step_losses, ".", color="0.7", markersize=3, label="loss per step")
    if len(step_losses) >= 2:
        window = min(max(5, steps_per_epoch // 10), len(step_losses))
        smooth = np.convolve(step_losses, np.ones(window) / window, mode="valid")
        ax.plot(range(window, len(step_losses) + 1), smooth, color="C0", label=f"average over {window} steps")
    for epoch in range(1, len(step_losses) // steps_per_epoch + 1):
        ax.axvline(epoch * steps_per_epoch + 0.5, color="0.9", zorder=0)
    ax.set_xlabel("training step" + (" (gray lines: epoch boundaries)" if total_steps > steps_per_epoch else ""))
    ax.set_ylabel("loss")
    ax.set_xlim(0, max(total_steps, len(step_losses)) + 1)
    ax.legend(loc="upper right")
    if title:
        ax.set_title(title, fontsize=10)
    fig.tight_layout()
    plot_handle.update(fig)

reuse = not RETRAIN and (FINETUNE_DIR / "unet_ema" / "config.json").is_file()
if reuse:
    # Earlier run: its weights, loss curve and record stay together in FINETUNE_DIR
    old_loss = LOSS_FILE if LOSS_FILE.is_file() else WORK_DIR / "training_loss.csv"   # runs before the record existed
    record = json.loads(RUN_RECORD.read_text()) if RUN_RECORD.is_file() else {}
    step_losses = pd.read_csv(old_loss)["loss"].tolist() if old_loss.is_file() else []
    steps_per_epoch = record.get("steps_per_epoch", len(step_losses) or 1)
    total_steps = len(step_losses)
    print("Reusing fine-tuned weights from an earlier run (set RETRAIN = True to train again):")
    print(json.dumps(record, indent=2) if record else "  (run details not recorded: run made before this notebook kept them)")
    redraw("earlier run" + (f", {record['finished']}" if record else ""))
else:
    print(f"{len(TRAIN_FILES)} images, batch size {BATCH_SIZE}: {steps_per_epoch} steps per epoch, "
          f"{total_steps} steps, learning rate warmup {warmup_steps} steps")
    if FINETUNE_DIR.exists():
        shutil.rmtree(FINETUNE_DIR)          # weights, loss and record are replaced together
    model = Model(checkpoint_dir=PROTICELLI_DIR / "checkpoint", vae_dir=PROTICELLI_DIR / "vae", device=DEVICE)
    torch.manual_seed(SEED)

    # Live loss plot. fit() reports each step's loss to its progress bar; a progress bar
    # that also records the loss lets the plot follow training without changing fit().
    class LossRecordingProgressBar(proticelli_training.tqdm):
        def set_postfix(self, *args, **kwargs):
            if "loss" in kwargs:
                step_losses.append(float(kwargs["loss"]))
                if len(step_losses) % max(1, steps_per_epoch // 5) == 0 or len(step_losses) == total_steps:
                    redraw()
            return super().set_postfix(*args, **kwargs)

    original_progress_bar = proticelli_training.tqdm
    proticelli_training.tqdm = LossRecordingProgressBar
    started = time.perf_counter()
    try:
        model.fit(
            image_dir=TRAIN_DIR,
            image_files=TRAIN_FILES,
            protein_names=TRAIN_PROTEINS,
            cell_line_names=TRAIN_CELL_LINES,
            output_dir=str(FINETUNE_DIR),
            num_epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            learning_rate=LEARNING_RATE,
            lr_warmup_steps=warmup_steps,
            save_model_epochs=10,
            checkpointing_steps=10**9,                    # no intermediate checkpoints (1.8 GB each)
            num_workers=4 if DEVICE == "cuda" else 0,     # worker processes are unreliable in notebooks on macOS
        )
    finally:
        proticelli_training.tqdm = original_progress_bar
    seconds = time.perf_counter() - started
    redraw()

    assert (FINETUNE_DIR / "unet_ema" / "config.json").is_file(), "fit() did not save weights"
    pd.Series(step_losses, name="loss").rename_axis("step").to_csv(LOSS_FILE)
    record = {"finished": datetime.datetime.now().strftime("%Y-%m-%d %H:%M"), "run_mode": RUN_MODE, "device": DEVICE,
              "training_cells": len(TRAIN_FILES), "proteins": len(set(TRAIN_PROTEINS)), "epochs": EPOCHS,
              "batch_size": BATCH_SIZE, "steps_per_epoch": steps_per_epoch, "steps": len(step_losses),
              "learning_rate": LEARNING_RATE, "minutes": round(seconds / 60, 1),
              "seconds_per_step": round(seconds / max(len(step_losses), 1), 1)}
    RUN_RECORD.write_text(json.dumps(record, indent=2))
    print(f"Training took {record['minutes']} minutes ({record['seconds_per_step']} s per step)")
    print("Fine-tuned weights saved in", FINETUNE_DIR / "unet_ema")
    del model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

## 6. Package, predict and evaluate

### Package the weights file

The new weights file has the same layout as the baseline's, so `blast_proticelli.wdl` and `run_inference.py` use it unchanged:

```
checkpoint/unet/                fine-tuned ProtiCelli weights
vae/                            ProtiCelli VAE (not trained here, copied as is)
reference/human_blast_db.zip    from section 3
reference/human_gene_map.zip    from section 3
```

It is built by `build_weights_zip.sh`, the same script used for the baseline's own weights files. On AnVIL, it is then copied to the workspace bucket, where a workflow can use it.

In [ ]:
import datetime

PACKAGE_DIR = WORK_DIR / "package"
if PACKAGE_DIR.exists():
    shutil.rmtree(PACKAGE_DIR)
shutil.copytree(FINETUNE_DIR / "unet_ema", PACKAGE_DIR / "checkpoint" / "unet")
shutil.copytree(PROTICELLI_DIR / "vae", PACKAGE_DIR / "vae")

stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d_%H%MUTC")
NEW_WEIGHTS = WORK_DIR / f"blast_proticelli_weights_finetuned_{RUN_MODE}_{stamp}.zip"
proc = subprocess.run(
    ["bash", str(TRAINING_SCRIPTS / "build_weights_zip.sh"), str(PACKAGE_DIR),
     str(BLAST_DB_DIR / "human_blast_db.zip"), str(GENE_MAP_DIR / "human_gene_map.zip"), str(NEW_WEIGHTS)],
    capture_output=True, text=True)
print(proc.stdout, proc.stderr)
proc.check_returncode()
shutil.rmtree(PACKAGE_DIR)
print(f"New weights file: {NEW_WEIGHTS} ({NEW_WEIGHTS.stat().st_size / 2**30:.1f} GiB)")

# On AnVIL, keep a copy in the workspace bucket: WORK_DIR is on the notebook's own disk
bucket = os.environ.get("WORKSPACE_BUCKET")
if bucket:
    dest = f"{bucket}/weights/{NEW_WEIGHTS.name}"
    subprocess.run(["gcloud", "storage", "cp", str(NEW_WEIGHTS), dest], check=True)
    print("Copied to", dest)
else:
    print("Not on AnVIL (no WORKSPACE_BUCKET): the weights file stays in WORK_DIR; copy it somewhere permanent.")

### Predict the held-out cells

The held-out cells go through the baseline's own prediction script, `run_inference.py`, exactly as a workflow would run a submission: three reference PNGs and the antibody's FASTA in, one predicted PNG out. It runs twice with the same random seed: once with the **baseline** weights (ProtiCelli as published) and once with the **fine-tuned** ones. Note that, as in the challenge, the protein is identified from the FASTA sequence (by BLAST), not taken from the manifest; `prediction_results.tsv` in each output folder records which protein was used.

`COMPARE_CELLS` sets how many held-out cells to predict, spread evenly over the held-out antibodies. Measured on an Apple M1-class laptop: about 1.6 minutes per set of weights for 12 cells at 20 diffusion steps (`PREDICTION_STEPS`), including loading the model.

In [ ]:
held_out = cells[cells["split"] == "test"]
if COMPARE_CELLS and len(held_out) > COMPARE_CELLS:
    per_antibody = -(-COMPARE_CELLS // held_out["antibody_id"].nunique())   # ceiling division
    # shuffle, then take the first few cells of each antibody (keeps every column)
    held_out = (held_out.sample(frac=1, random_state=SEED)
                .groupby("antibody_id").head(per_antibody).head(COMPARE_CELLS))

records = []
for _, row in held_out.iterrows():
    f = cell_files(row)
    records.append({"prediction_id": row["image_key"], "microtubules_image": str(f["microtubules"]),
                    "nucleus_image": str(f["nucleus"]), "er_image": str(f["er"]), "fasta_file": str(f["sequence"])})
inputs_json = WORK_DIR / "held_out_inputs.json"
inputs_json.write_text(json.dumps(records, indent=2))
print(f"{len(records)} held-out cells from {held_out['antibody_id'].nunique()} antibodies")

BASELINE_WEIGHTS = Path(EXISTING_WEIGHTS_ZIP).expanduser() if EXISTING_WEIGHTS_ZIP else WORK_DIR / "blast_proticelli_weights_baseline.zip"
if not BASELINE_WEIGHTS.exists():
    print("Building the baseline weights file from the original ProtiCelli weights (several minutes)...", end=" ", flush=True)
    started = time.perf_counter()
    subprocess.run(["bash", str(TRAINING_SCRIPTS / "build_weights_zip.sh"), str(PROTICELLI_DIR),
                    str(BLAST_DB_DIR / "human_blast_db.zip"), str(GENE_MAP_DIR / "human_gene_map.zip"),
                    str(BASELINE_WEIGHTS)], check=True, capture_output=True)
    print(f"{(time.perf_counter() - started) / 60:.1f} minutes")

def predict(weights_zip, name):
    out = WORK_DIR / "predictions" / name
    cmd = [sys.executable, str(REPO_DIR / "run_inference.py"), "--inputs", str(inputs_json),
           "--weights_zip", str(weights_zip), "--output_dir", str(out),
           "--weights_dir", str(WORK_DIR / "inference_weights" / name),
           "--reference_data_dir", str(WORK_DIR / "inference_reference" / name),
           "--num_inference_steps", str(PREDICTION_STEPS), "--seed", str(SEED)]
    print(f"Predicting with the {name} weights...", end=" ", flush=True)
    started = time.perf_counter()
    proc = subprocess.run(cmd, capture_output=True, text=True, cwd=WORK_DIR)
    if proc.returncode != 0:
        raise RuntimeError(proc.stdout[-3000:] + proc.stderr[-3000:])
    print(f"{(time.perf_counter() - started) / 60:.1f} minutes")
    return out

pred_dirs = {"baseline": predict(BASELINE_WEIGHTS, "baseline"), "fine-tuned": predict(NEW_WEIGHTS, "finetuned")}
used = pd.read_csv(pred_dirs["fine-tuned"] / "prediction_results.tsv", sep="\t", dtype=str)
held_out = held_out.merge(used[["prediction_id", "status", "protein_passed_to_model"]],
                          left_on="image_key", right_on="prediction_id", how="left")
print(held_out.drop_duplicates("antibody_id")[["antibody_id", "gene_name", "protein_passed_to_model", "status"]].to_string(index=False))

### Evaluate

The predictions are scored against the real protein images with **cm4ai-evaluate**, the challenge's local evaluation tool, inside each cell's mask and on the whole image. Its metrics are drafts: the evaluation team is still deciding which the challenge will use and how to combine them, so treat these numbers as a demonstration of the evaluation step, not as the challenge score. No single metric is enough: what each one measures, and which direction is better, is in the [cm4ai-evaluate metric primer](https://github.com/cm4ai-anvil-dream-challenge/cm4ai-evaluate/blob/main/docs/metrics.md).

With `"quick"` settings, and with training data that overlaps ProtiCelli's own, expect small differences between the baseline and fine-tuned weights, mostly noise.

While the cm4ai-evaluate repository is private, install it once from a terminal (where git can ask for credentials) into this kernel's Python, `sys.executable`: `<that path> -m pip install "cm4ai-evaluate[extended] @ git+https://github.com/cm4ai-anvil-dream-challenge/cm4ai-evaluate.git"`.

In [ ]:
def find_evaluate():
    # On PATH, or next to this kernel's Python (an install from a terminal into the kernel's environment)
    beside = Path(sys.executable).parent / "cm4ai-evaluate"
    return shutil.which("cm4ai-evaluate") or (str(beside) if beside.is_file() else None)

if not find_evaluate():
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                    "cm4ai-evaluate[extended] @ git+https://github.com/cm4ai-anvil-dream-challenge/cm4ai-evaluate.git"], check=True)
evaluate_cmd = find_evaluate()
assert evaluate_cmd, "cm4ai-evaluate is not installed; see the note above this cell"

EVAL_DIR = WORK_DIR / "evaluation"
EVAL_DIR.mkdir(parents=True, exist_ok=True)
scores = {}
for name, d in pred_dirs.items():
    eval_manifest = EVAL_DIR / f"{name}_manifest.csv"
    pd.DataFrame({"prediction_id": held_out["image_key"],
                  "prediction": [str(d / f"{k}.png") for k in held_out["image_key"]],
                  "ground_truth": [str(cell_files(r)["protein"]) for _, r in held_out.iterrows()],
                  "mask": [str(cell_files(r)["mask"]) for _, r in held_out.iterrows()]}).to_csv(eval_manifest, index=False)
    proc = subprocess.run([evaluate_cmd, str(eval_manifest), "-o", str(EVAL_DIR / name)], capture_output=True, text=True)
    if proc.returncode == 2:
        raise RuntimeError(proc.stdout[-2000:] + proc.stderr[-2000:])
    scores[name] = pd.read_csv(EVAL_DIR / f"{name}.csv").set_index("prediction_id")
print("Scores in", EVAL_DIR)

SHOW_METRICS = ["masked_pearson_r", "masked_cosine_similarity", "masked_ms_ssim", "masked_mean_absolute_error"]
SHOW_METRICS = [m for m in SHOW_METRICS if m in scores["baseline"].columns]
summary = pd.DataFrame({name: s[SHOW_METRICS].mean() for name, s in scores.items()})
summary["change"] = summary["fine-tuned"] - summary["baseline"]
print("\nMean over held-out cells (mean absolute error: lower is better; the others: higher is better)")
summary.round(3)

In [ ]:
metric = "masked_pearson_r"
paired = pd.DataFrame({name: s[metric] for name, s in scores.items()}).join(
    held_out.set_index("image_key")[["gene_name"]])

fig, ax = plt.subplots(figsize=(5, 5))
for gene, g in paired.groupby("gene_name"):
    ax.scatter(g["baseline"], g["fine-tuned"], label=gene, s=30)
lims = [min(paired[["baseline", "fine-tuned"]].min().min(), 0), 1]
ax.plot(lims, lims, color="0.6", linewidth=1)
ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_xlabel(f"baseline model: {metric}")
ax.set_ylabel(f"fine-tuned model: {metric}")
ax.set_title("Each held-out cell; above the line means fine-tuning helped", fontsize=10)
ax.legend(fontsize=8, title="protein")
plt.show()

In [ ]:
show = held_out.groupby("antibody_id", group_keys=False).head(1)["image_key"].tolist()[:4]
fig, axes = plt.subplots(len(show), 4, figsize=(13, 3.4 * len(show)), squeeze=False)
for row_axes, key in zip(axes, show):
    r = held_out[held_out["image_key"] == key].iloc[0]
    f = cell_files(r)
    truth = imread(f["protein"])
    preds = {name: imread(pred_dirs[name] / f"{key}.png") for name in pred_dirs}
    top = max(np.percentile(truth, 99.9), 1)      # one scale for the real image and both predictions
    panels = [(f"reference channels\n{key}", reference_composite(imread(f["microtubules"]), imread(f["nucleus"]), imread(f["er"]))),
              (f"real (ground truth)\n{r['gene_name']}", protein_green(truth, top))]
    panels += [(f"{name} model prediction\nr = {scores[name].loc[key, metric]:.2f}", protein_green(p, top)) for name, p in preds.items()]
    for ax, (title, image) in zip(row_axes, panels):
        ax.imshow(image)
        ax.contour(imread(f["mask"]) > 0, levels=[0.5], colors="white", linewidths=0.6)
        ax.set_title(title, fontsize=9)
        ax.axis("off")
fig.suptitle(f"Held-out cells: the real protein and both models' predictions, on one brightness scale (r: {metric})")
plt.tight_layout()
plt.show()

## Next steps

* **Use the weights file in a submission.** Pass the copy in the workspace bucket (`weights/` folder) as `weights_file` to `blast_proticelli.wdl`; nothing else in the workflow changes.
* **Where a team could go further:** train for longer or on more data; tune normalization or augmentation in section 4; add proteins to ProtiCelli's vocabulary (section 4); replace the BLAST step with a model that reads the amino acid sequence directly; or swap ProtiCelli for another model entirely, keeping the same inputs and outputs.
* **Housekeeping.** `WORK_DIR` holds several GB (weights copies, training images); delete it when done.